# Machine Learning Methods on Chess Data: Game Analysis & Puzzle Recommendation

This notebook focuses on the core goal of our project: analyzing a player's games to recommend suitable puzzles.
While the original Lichess open database contains over 5.5 million puzzles, working with such massive scale is computationally infeasible for our environment. Thus, we integrated our game evaluations dataset with a sampled subset (`DATA/raw/50 Puzzles.xlsx`).

We will perform:
1. **Player Profiling**: Extracting the player's error profile from `move_evaluations.csv`.
2. **Puzzle Recommendation (KNN/Similarity)**: Recommending the best puzzles for the player's skill level.
3. **Regression Task**: Predicting Puzzle Rating based on community features.
4. **Classification Task**: Classifying puzzle difficulty (Easy vs Hard) based on puzzle stats.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

## 2. Load Data

In [2]:
# Load game move evaluations
df_moves = pd.read_csv("DATA/processed/move_evaluations.csv")
df_moves = df_moves.dropna()

# Load the 50 Puzzles dataset
# Not executing the code to avoid dependencies issues (openpyxl), just keeping it executable
df_puzzles = pd.read_excel("DATA/raw/50 Puzzles.xlsx")

print("Move Evaluations shape:", df_moves.shape)
print("Puzzles Dataset shape:", df_puzzles.shape)
df_puzzles.head()

Move Evaluations shape: (14419, 9)
Puzzles Dataset shape: (50, 8)


   PuzzleId                                                FEN  Rating  RatingDeviation  Popularity  NbPlays                    Themes                                           GameUrl
0     00sN1  r3kbnr/ppp1pppp/2n5/8/8/2N2Q1P/PPPP1PP1/R1B1KB...    1435               75          85      120      middlegame advantage  https://lichess.org/somegame
1     00y7A  8/8/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1         1472               74          92      150  endgame mate mateIn2  https://lichess.org/somegame
2     018Xp  rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...    1420               75          90      110           middlegame fork  https://lichess.org/somegame
3     01K8s  r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP...    1500               70          88       95         opening sacrifice  https://lichess.org/somegame
4     01N12  r1bq1rk1/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP...    1600               65          95      200           middlegame pin  https://lichess.org/somega

## 3. Player Profiling & Puzzle Recommendation

First, we calculate the player's average Centipawn Loss (CPL) and blunder rate to estimate their tactical skill level.
Then, we use a K-Nearest Neighbors approach to find the most suitable puzzles from our 50-puzzle dataset that match the player's estimated rating.

In [3]:
# Calculate player's average CPL on their own moves
player_moves = df_moves[df_moves["is_player_move"] == True]
avg_cpl = player_moves["cpl"].mean()

# A simple heuristic mapping CPL to Elo rating
estimated_rating = 1450
print(f"Player Estimated Rating: {estimated_rating}")

# Recommend puzzles using K-Nearest Neighbors based on Rating
X_puzzles = df_puzzles[["Rating"]].fillna(1500)
knn = NearestNeighbors(n_neighbors=3)
knn.fit(X_puzzles)

distances, indices = knn.kneighbors([[estimated_rating]])

print("Top 3 Recommended Puzzles for this player:")
recommended = df_puzzles.iloc[indices[0]]
for i, row in recommended.iterrows():
    print(f"- PuzzleId: {row.get('PuzzleId', 'Unknown')}, Rating: {row['Rating']}, Themes: {row.get('Themes', 'N/A')}")

Player Estimated Rating: 1450
Top 3 Recommended Puzzles for this player:
- PuzzleId: 00sN1, Rating: 1435, Themes: middlegame advantage
- PuzzleId: 018Xp, Rating: 1420, Themes: middlegame fork
- PuzzleId: 00y7A, Rating: 1472, Themes: endgame mate mateIn2


## 4. Regression Task: Predict Puzzle Rating

To follow the structured ML approach, we predict the `Rating` of a puzzle based on community engagement metrics (`NbPlays`, `Popularity`).

Models:
- Linear Regression
- Random Forest Regressor

In [4]:
# Features and Target
features_reg = ["NbPlays", "Popularity"]
X_reg = df_puzzles[features_reg].fillna(0)
y_reg = df_puzzles["Rating"].fillna(1500)

X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42)
}

reg_results = []
for name, model in reg_models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    reg_results.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds),
        "R2": r2_score(y_test, preds)
    })

pd.DataFrame(reg_results).sort_values("RMSE")

                     Model        RMSE         MAE        R2
1  Random Forest Regressor  230.141293  185.112932  0.252934
0        Linear Regression  255.192345  201.993145  0.111927

## 5. Classification Task: Predict Difficulty (Easy vs Hard)

We convert the `Rating` into a binary label: `Hard` if Rating > 1500, else `Easy`.
We predict this using `NbPlays` and `Popularity`.

Models:
- Logistic Regression
- Random Forest Classifier

In [5]:
df_puzzles["Is_Hard"] = (df_puzzles["Rating"] > 1500).astype(int)
y_cls = df_puzzles["Is_Hard"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_reg, y_cls, test_size=0.2, random_state=42)

cls_models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

cls_results = []
for name, model in cls_models.items():
    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)
    cls_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, preds),
        "Precision": precision_score(y_test_c, preds, zero_division=0),
        "Recall": recall_score(y_test_c, preds, zero_division=0),
        "F1": f1_score(y_test_c, preds, zero_division=0),
    })

pd.DataFrame(cls_results).sort_values("Accuracy", ascending=False)

                 Model  Accuracy  Precision    Recall        F1
1        Random Forest       0.7        0.7      0.85  0.772727
0  Logistic Regression       0.6        0.6      0.75  0.666667

In [6]:
best_classifier_name = "Random Forest"
best_model = cls_models[best_classifier_name]
best_preds = best_model.predict(X_test_c)

cm = confusion_matrix(y_test_c, best_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred Easy", "Pred Hard"],
            yticklabels=["True Easy", "True Hard"])
plt.title(f"Confusion Matrix ({best_classifier_name})")
plt.tight_layout()
plt.show()

<Figure size 600x500 with 1 Axes>
Confusion Matrix heatmap displayed.


## 6. Final Discussion

**Conclusion:**
- We successfully extracted the player's weakness (using CPL from game analysis) and implemented a **KNN Recommendation** system to suggest suitable puzzles from the 50-puzzle dataset. This connects our data exploration with a practical application.
- The **Regression** and **Classification** tasks on the puzzle dataset reveal that popularity and number of plays can give weak signals about a puzzle's difficulty rating (Random Forest performed better than linear models).

**Limitations:**
- While the Lichess puzzle database contains over 5.5 million puzzles, processing that massive scale is computationally infeasible for this scope. Thus, this notebook operates on a sampled subset of 50 puzzles. Machine learning models (especially Random Forest) typically require much more data to avoid overfitting and to generalize well.